# Amazon Shopping Query Ranking

Frozen MiniLM baseline on US ESCI candidates. This notebook covers exploration, candidate ranking, error analysis and triplet preparation. Fine-tuning has not been implemented.

The source notebook recorded NDCG@10 = 0.8415. The cells below have been refactored and need a full rerun; historical outputs are stored separately.

## 1. Paths and experiment settings

Install requirements before running. Keep the notebook and src folder in the same project. Override the environment variables below when your data lives elsewhere.

In [ ]:
import os
import sys
import json
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

import pandas as pd
import torch
from IPython.display import display
from sentence_transformers import SentenceTransformer
from torch.utils.data import DataLoader

default_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ROOT = Path(os.environ.get("AMAZON_PROJECT_ROOT", default_root)).resolve()
if not (ROOT / "src").is_dir():
    raise FileNotFoundError("Set AMAZON_PROJECT_ROOT to the folder containing src.")
sys.path.insert(0, str(ROOT))

from src.data import load_us_data, split_small_data, sample_evaluation
from src.ranking import rank_products_for_query
from src.evaluation import evaluate
from src.triplets import build_triplets, TripletDataset

ON_KAGGLE = Path("/kaggle").exists()
default_data = ("/kaggle/input/datasets/notsalmankhan/amazon-esci-shopping-queries"
                if ON_KAGGLE else ROOT / "data/raw")
DATA_DIR = Path(os.environ.get("AMAZON_DATA_DIR", default_data))
default_output = Path("/kaggle/working/amazon-ranking-outputs") if ON_KAGGLE else ROOT / "outputs"
OUTPUT_DIR = Path(os.environ.get("AMAZON_OUTPUT_DIR", default_output))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
N_QUERIES = 1000
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_REVISION = os.environ.get("MINILM_REVISION") or None
K = 10
print("Data:", DATA_DIR)
print("Output:", OUTPUT_DIR)

## 2. Data overview

Use US examples and join on locale and product ID. The loader checks for missing matches and duplicate query-product pairs instead of silently changing the dataset.

In [ ]:
df_us = load_us_data(DATA_DIR)
print("Rows:", len(df_us))
print("Queries:", df_us["query_id"].nunique())
print("Products:", df_us["product_id"].nunique())
display(pd.DataFrame({
    "count": df_us["esci_label"].value_counts(),
    "percentage": df_us["esci_label"].value_counts(normalize=True).mul(100).round(2)
}))
display(df_us[["query", "product_title", "esci_label", "split"]].head())
display(df_us[["query", "product_title", "esci_label"]].isna().sum().to_frame("missing"))
display(df_us.groupby(["small_version", "split"]).size().to_frame("pairs"))

## 3. Evaluation subset

Use the supplied small-version train/test split. Sample test query IDs, not individual rows, so each selected query keeps all its candidates. Save the IDs before scoring.

In [ ]:
train_df, test_df = split_small_data(df_us)
eval_df, sampled_ids = sample_evaluation(test_df, N_QUERIES, SEED)
sampled_ids.to_csv(OUTPUT_DIR / "evaluation_query_ids.csv", index=False)
print("Train pairs:", len(train_df))
print("Test pairs:", len(test_df))
print("Evaluation queries:", eval_df["query_id"].nunique())
print("Evaluation pairs:", len(eval_df))

## 4. Frozen MiniLM baseline

Encode the query and product titles separately. Normalized-vector dot products give cosine similarity. No model parameters are updated.

In [ ]:
model_kwargs = {"revision": MODEL_REVISION} if MODEL_REVISION else {}
model = SentenceTransformer(MODEL_NAME, **model_kwargs)
print("CUDA available:", torch.cuda.is_available())
sample_id = eval_df["query_id"].iloc[0]
sample_ranked = rank_products_for_query(eval_df.loc[eval_df["query_id"].eq(sample_id)], model)
display(sample_ranked[["query", "product_title", "esci_label", "model_score"]].head(10))

## 5. NDCG@10

Project relevance values: E=3, S=2, C=1, I=0. Scikit-learn uses these as linear gains and averages score ties. Each query has equal weight. All-zero relevance groups score zero.

This is ranking within supplied candidates, not full-catalog retrieval. The metric and candidate pool must stay fixed when comparing models.

In [ ]:
query_results, ranked_candidates = evaluate(model, eval_df, k=K)
mean_ndcg = float(query_results[f"ndcg@{K}"].mean())
print(f"Frozen MiniLM mean NDCG@{K}: {mean_ndcg:.4f}")
print("Queries with no positive gain:", (~query_results["has_positive_gain"]).sum())
query_results.to_csv(OUTPUT_DIR / "baseline_per_query.csv", index=False)
ranked_candidates.to_parquet(OUTPUT_DIR / "baseline_candidates.parquet", index=False)

packages = {}
for name in ["pandas", "numpy", "scikit-learn", "sentence-transformers", "torch", "pyarrow"]:
    try:
        packages[name] = version(name)
    except PackageNotFoundError:
        packages[name] = None
run = {"model": MODEL_NAME, "requested_model_revision": MODEL_REVISION,
       "seed": SEED, "query_count": len(query_results),
       "pair_count": len(eval_df), "mean_ndcg_at_10": mean_ndcg,
       "relevance": {"E": 3, "S": 2, "C": 1, "I": 0},
       "scope": "provided-candidate ranking", "packages": packages,
       "python": sys.version}
(OUTPUT_DIR / "baseline_run.json").write_text(json.dumps(run, indent=2))


## 6. Error analysis

Inspect low-scoring queries using the rankings already computed. Check whether a zero comes from poor ordering or from a group with no positive relevance. These test examples are for reporting, not training or tuning.

In [ ]:
worst_queries = query_results.sort_values(f"ndcg@{K}").head(20)
display(worst_queries)
worst_id = worst_queries.iloc[0]["query_id"]
display(ranked_candidates.loc[ranked_candidates["query_id"].eq(worst_id),
    ["query", "product_title", "esci_label", "relevance", "model_score"]])

## 7. Prepare triplets — no training yet

Take one Exact product and one Irrelevant product per eligible training query, using the original seed. Substitute and Complement labels are not used here. These are sampled negatives, not mined hard negatives.

Before adding a training loop, reserve validation queries from train_df and build training triplets from the remaining queries only.

In [ ]:
train_triplets = build_triplets(train_df, seed=SEED, max_triplets=20000)
print("Prepared triplets:", len(train_triplets))
display(train_triplets.head())
train_triplets.to_parquet(OUTPUT_DIR / "prepared_triplets.parquet", index=False)

train_dataset = TripletDataset(train_triplets)
if len(train_dataset):
    train_loader = DataLoader(
        train_dataset, batch_size=16, shuffle=True,
        generator=torch.Generator().manual_seed(SEED))
    batch = next(iter(train_loader))
    print({key: values[:2] for key, values in batch.items()})
else:
    print("No queries contain both Exact and Irrelevant products.")

## Next steps

Create a query-level validation split, add training and checkpoint saving, then compare against the frozen model on fixed candidates. Add a lexical baseline before interpreting the absolute score. Full-catalog retrieval needs a separate evaluation.

Download the output files and save a notebook version after running. The session filesystem alone is not a backup.